# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SymbolPamnani/Flyrank-ML-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Answer

**Unit of analysis:** one pseudonymized content item/page.

The dataset contains one row per content item, identified by `content_id`. The available search-performance measurements are aggregated over defined reporting windows rather than representing individual search events.

The main reporting window available in this dataset is the 90-day period represented by fields such as `impressions_90d`, `clicks_90d`, and `sessions_90d`. The dataset also contains last-30-day and previous-30-day measurements, which can be used to compare recent performance with the preceding period.

The exact time boundaries are verified from the available fields rather than assumed from the dataset.

In [1]:
import pandas as pd

DATA_PATH = "content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)

print("\nRows per content_id:")
print(df["content_id"].nunique())

print("\nRows per client_id:")
print(df["client_id"].nunique())

print("\n90-day fields:")
print([
    col for col in df.columns
    if "_90d" in col
])

print("\n30-day fields:")
print([
    col for col in df.columns
    if "_last_30d" in col or "_prev_30d" in col
])

Shape: (30000, 44)

Rows per content_id:
30000

Rows per client_id:
32

90-day fields:
['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d']

30-day fields:
['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Answer

### Features

The candidate features are measurable signals that could be available to the model when producing a ranking:

- `search_volume`
- `competition`
- `cpc`
- `content_type`
- `main_intent`
- `word_count`
- `char_count`
- `impressions_last_30d`
- `clicks_last_30d`
- `sessions_last_30d`
- `impressions_prev_30d`
- `clicks_prev_30d`
- `sessions_prev_30d`
- `content_age_days`
- `days_since_last_update`
- `ctr`
- `avg_position`
- `engagement_rate`
- `scroll_rate`
- `ai_traffic_pct`

### Label / proxy

`trend_direction` is used to define the declining proxy for the current framing task:

`declining_proxy = 1` when `trend_direction == "down"`, otherwise `0`.

This is a defined proxy based on the available dataset rather than an independently observed future outcome.

### Context

The following fields provide context or identifiers but are not intended as predictive features:

- `content_id`
- `client_id`
- `provider_used`
- `model_used`
- `competition_level`
- `age_tier`
- `age_tier_order`
- `freshness_tier`
- `word_count_tier`
- `char_count_tier`
- `impression_tier`
- `position_tier`

### Excluded

The raw 90-day aggregate outcome fields are excluded from the feature set when they represent the same outcome period used to define the target or would make the model depend directly on the outcome being predicted.

This includes fields such as:

- `impressions_90d`
- `clicks_90d`
- `pageviews_90d`
- `sessions_90d`
- `users_90d`
- `engaged_sessions_90d`
- `ai_sessions_90d`
- `scroll_events_90d`
- `days_with_impressions`
- `days_with_sessions`

The exact exclusions will be checked against the available schema before modeling.

In [2]:
feature_columns = [
    "search_volume",
    "competition",
    "cpc",
    "content_type",
    "main_intent",
    "word_count",
    "char_count",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

context_columns = [
    "content_id",
    "client_id",
    "provider_used",
    "model_used",
    "competition_level",
    "age_tier",
    "age_tier_order",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier"
]

excluded_columns = [
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions"
]

print("FEATURES")
print([col for col in feature_columns if col in df.columns])

print("\nCONTEXT")
print([col for col in context_columns if col in df.columns])

print("\nEXCLUDED")
print([col for col in excluded_columns if col in df.columns])

print("\nMissing from planned fields:")
planned = feature_columns + context_columns + excluded_columns
print([col for col in planned if col not in df.columns])

FEATURES
['search_volume', 'competition', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

CONTEXT
['content_id', 'client_id', 'provider_used', 'model_used', 'competition_level', 'age_tier', 'age_tier_order', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']

EXCLUDED
['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions']

Missing from planned fields:
[]


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## 3. Answer

The checks below verify the grain of the data, row and content counts, missing values, and the availability of the reporting-window fields.

The dataset should contain one row per `content_id`. Missingness is checked for the fields used by the contract. The available 90-day and 30-day fields are also inspected to confirm the reporting windows represented in the dataset.

In [3]:
print("Total rows:", len(df))
print("Unique content items:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())

print("\nDuplicate content_id rows:", df["content_id"].duplicated().sum())

print("\nMissing values in planned fields:")

check_columns = [
    col for col in feature_columns
    if col in df.columns
]

missing = df[check_columns].isnull().sum()

print(missing[missing > 0])

print("\n90-day fields:")
for col in df.columns:
    if "_90d" in col:
        print("-", col)

print("\nLast/previous 30-day fields:")
for col in df.columns:
    if "_last_30d" in col or "_prev_30d" in col:
        print("-", col)

Total rows: 30000
Unique content items: 30000
Unique clients: 32

Duplicate content_id rows: 0

Missing values in planned fields:
search_volume    2468
competition      2468
cpc              2468
main_intent      2374
word_count       7699
char_count       7699
scroll_rate       125
dtype: int64

90-day fields:
- impressions_90d
- clicks_90d
- pageviews_90d
- sessions_90d
- users_90d
- engaged_sessions_90d
- ai_sessions_90d
- scroll_events_90d

Last/previous 30-day fields:
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Answer

This dataset has several important limits.

First, the data is aggregated rather than event-level, so it cannot explain individual user searches or individual clicks.

Second, the available trend label is a defined proxy based on the supplied data rather than a guaranteed future outcome. A declining label therefore should not be interpreted as proof that a page will continue declining.

Third, the dataset contains different reporting windows, including 90-day aggregates and last-30-day versus previous-30-day measurements. These windows can overlap, so they should not be treated as independent observations.

Finally, the dataset is anonymized and does not provide all possible external factors that could affect content performance. Therefore, the model can support prioritization and investigation but cannot establish causation.

In [4]:
print("Reporting-window fields:")

window_groups = {
    "90-day": [col for col in df.columns if "_90d" in col],
    "last 30-day": [col for col in df.columns if "_last_30d" in col],
    "previous 30-day": [col for col in df.columns if "_prev_30d" in col]
}

for window, columns in window_groups.items():
    print(f"\n{window}:")
    for col in columns:
        print("-", col)

print("\nTrend categories:")
print(df["trend_direction"].value_counts())

print("\nContent age range:")
print(
    "Minimum:", df["content_age_days"].min(),
    "| Maximum:", df["content_age_days"].max()
)

Reporting-window fields:

90-day:
- impressions_90d
- clicks_90d
- pageviews_90d
- sessions_90d
- users_90d
- engaged_sessions_90d
- ai_sessions_90d
- scroll_events_90d

last 30-day:
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d

previous 30-day:
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d

Trend categories:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Content age range:
Minimum: 90 | Maximum: 564


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.